# Qwen3.8-27B: endpoint chuẩn OpenAI trên Kaggle 2×T4

**Run All** → dò máy → lấy `llama-server` dựng sẵn (không biên dịch) → tải model GGUF → chạy model → tự kiểm (trả lời, nghĩ, gọi tool, đọc ảnh, tốc độ) → mở gateway + tunnel → in khung **ENDPOINT SẴN SÀNG**.

Colab Pro dùng notebook riêng cho từng loại máy: `Qwen3.8-27B A100`, `Qwen3.8-27B G4` (thư mục `Colab/`).

| Bước | Việc cần làm |
|---|---|
| 1 | Kaggle: cột phải → Settings → Accelerator **GPU T4 x2**, Internet **On** |
| 2 | **Run All**, chờ khung `ENDPOINT SẴN SÀNG` ở ô 9 (lần đầu khoảng 10 phút, phần lớn là tải model) |
| 3 | Chép `Base URL`, `API key`, `Model` vào chương trình cần dùng. |

**Endpoint mở đúng 4 đường** (mọi đường đều cần `Authorization: Bearer <API key>`): `GET /healthz`, `GET /capabilities`, `GET /v1/models`, `POST /v1/chat/completions`.

**Sơ đồ:** `cloudflared → 127.0.0.1:8787 gateway → 127.0.0.1:8080 llama-server`. Chỉ gateway ra ngoài; `llama-server` chỉ nghe trong máy.

**Muốn địa chỉ và khóa không đổi giữa các phiên** (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khóa):

| Secret | Tác dụng | Không đặt thì |
|---|---|---|
| `GATEWAY_TOKEN` | API key cố định (tự đặt một chuỗi dài, khó đoán) | mỗi phiên sinh khóa mới, in ở khung |
| `CF_TUNNEL_TOKEN` + `PUBLIC_BASE_URL` | tunnel có tên của Cloudflare, địa chỉ cố định (cần một tên miền trên Cloudflare) | Quick Tunnel, địa chỉ `trycloudflare.com` đổi mỗi phiên |
| `HF_TOKEN` | chỉ cần khi repo model đòi đăng nhập | tải ẩn danh |

In [ ]:
# 1) Dò máy: nền tảng, GPU, đĩa, mạng. Ô này không đoán cấu hình; các ô sau rẽ nhánh theo số đo ở đây.
import base64, hashlib, importlib.util, io, json, os, re, secrets, shutil, site, subprocess, sys, tarfile, time
import urllib.error, urllib.request
from pathlib import Path

NOTEBOOK_VERSION = "2026-10-13"

# Kiểm Kaggle TRƯỚC: máy Kaggle dựng từ image Colab nên cũng có biến COLAB_* và gói google-colab.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = not ON_KAGGLE and importlib.util.find_spec("google.colab") is not None
CHAY_NEN = ON_KAGGLE and os.environ["KAGGLE_KERNEL_RUN_TYPE"] != "Interactive"   # Save & Run All hoặc đẩy bằng CLI

WORK = Path("/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "work").resolve()
# File nặng (model, llama.cpp) để ngoài WORK: /kaggle/working chỉ 19,5 GiB và bị lưu thành Output.
SCRATCH = (Path("/tmp") if ON_KAGGLE else WORK) / "qwen38"
MODEL_DIR, BIN_DIR, LOG_DIR = SCRATCH / "models", SCRATCH / "bin", WORK / "qwen38_logs"
for _p in (MODEL_DIR, BIN_DIR, LOG_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def nvidia_smi(*args):
    exe = shutil.which("nvidia-smi") or next(
        (p for p in ("/opt/bin/nvidia-smi", "/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi") if Path(p).exists()), None)
    if not exe:
        return ""
    r = subprocess.run([exe, *args], text=True, capture_output=True)
    return r.stdout.strip() if r.returncode == 0 else ""


def gpu_csv(fields):
    out = nvidia_smi(f"--query-gpu={fields}", "--format=csv,noheader,nounits")
    return [[x.strip() for x in line.split(",")] for line in out.splitlines() if line.strip()]


GPUS = [{"index": int(i), "name": n, "gib": int(m) / 1024} for i, n, m in gpu_csv("index,name,memory.total")]
if not GPUS:
    raise SystemExit(
        "Không thấy GPU NVIDIA.\n"
        "  Kaggle: cột phải -> Settings -> Accelerator -> GPU T4 x2, rồi Run All lại.\n"
        "  Colab Pro: Runtime -> Change runtime type -> A100 hoặc G4.")

try:
    urllib.request.urlopen("https://huggingface.co/api/models/Qwen/Qwen3.8-27B", timeout=20).read(64)
except urllib.error.HTTPError:
    pass                                    # có trả lời HTTP là có mạng
except Exception as e:
    raise SystemExit(f"Chưa có mạng ({type(e).__name__}: {e}).\n"
                     "  Kaggle: Settings -> Internet -> On (tài khoản cần xác minh số điện thoại).")

print(json.dumps({
    "notebook": NOTEBOOK_VERSION, "kaggle": ON_KAGGLE, "colab": ON_COLAB, "chay_nen": CHAY_NEN,
    "python": sys.version.split()[0], "cpu": os.cpu_count(),
    "gpu": [f"{g['name']} {g['gib']:.1f} GiB" for g in GPUS],
    "dia_trong_gib": {str(SCRATCH): round(shutil.disk_usage(SCRATCH).free / 2**30, 1),
                      str(WORK): round(shutil.disk_usage(WORK).free / 2**30, 1)},
}, ensure_ascii=False, indent=1))

In [ ]:
# 2) CẤU HÌNH: chỉ cần sửa ô này. Để "auto" thì notebook tự chọn theo VRAM đo được ở ô 1.
MODEL = "goc"            # "goc": Qwen3.8-27B chính gốc (Apache-2.0) | "swift15": Swift-1.5 (giấy phép riêng của tác giả)
QUANT = "auto"           # "auto" hoặc tên mức nén trong bảng MODELS: "Q4_K_M", "Q5_K_M", "Q6_K"...
CTX = "auto"             # "auto" hoặc số token ngữ cảnh: 32768, 65536, 131072, 262144
KV = "auto"              # nén bộ nhớ ngữ cảnh: "auto" (= q8_0) | "f16" | "q8_0" | "q4_0"
VISION = True            # nạp phần đọc ảnh (thêm khoảng 0,9 GiB VRAM)
THINKING = True          # mặc định có nghĩ; từng lượt gọi vẫn tự bật tắt được (xem khung ENDPOINT)
MTP = True               # đoán trước vài token bằng đầu MTP rồi kiểm (kết quả không đổi, chỉ nhanh hơn; thêm khoảng 2 GiB VRAM)
MTP_N = 3                # số token đoán trước mỗi bước
SPLIT = "auto"           # cách chia model cho 2 GPU: "auto" (thử "tensor", hỏng thì "layer") | "tensor" | "layer"
TUNNEL = "auto"          # "auto": Kaggle mở tunnel, Colab không mở (xem ô 9) | True | False
KEEP_ALIVE_HOURS = 4     # ô 10 giữ phiên bấy nhiêu giờ rồi tự tắt; thời gian chạy trừ vào hạn mức GPU
KIEM_CHAT_LUONG = False   # True: ô 9b tự kiểm chất lượng trước ô 10 (khoảng 5–10 phút; app gọi lúc đó sẽ chậm hơn)
LLAMA_TAG = "b11429"     # bản llama.cpp dựng sẵn (b11429 = bản ổn định v0.6.0, 05/10/2026)

ENGINE_PORT = 8080       # llama-server, chỉ nghe 127.0.0.1
PORT_GW = 8787           # gateway, cổng DUY NHẤT đưa ra tunnel

# Cỡ file tính bằng GB (10^9 byte), đọc từ API Hugging Face ngày 13/10/2026.
MODELS = {
    "goc": {
        "repo": "unsloth/Qwen3.8-27B-GGUF", "alias": "qwen3.8-27b",
        "file": "Qwen3.8-27B-UD-{q}.gguf", "mmproj": "mmproj-F16.gguf",
        "quants": {"Q4_K_M": 16.46, "Q5_K_M": 19.77, "Q6_K": 21.98, "Q8_K_L": 28.05},
    },
    "swift15": {
        "repo": "ukisai/Swift-1.5-Qwen3.8-27B-GGUF", "alias": "swift-1.5-qwen3.8-27b",
        "file": "Swift-1.5-Qwen3.8-27B-{q}.gguf", "mmproj": "mmproj-Swift-1.5-Qwen3.8-27B-F16.gguf",
        "quants": {"Q4_K_M": 17.44, "Q5_K_M": 20.92, "Q6_K": 23.86, "Q8_0": 29.05},
    },
}
MMPROJ_GB = 0.93
# Đầu MTP lượng tử Q4_0 tách riêng của Unsloth (từ bản gốc; Swift-1.5 giữ nguyên đầu MTP của bản gốc theo model card).
# Đo 13/10 trên 2×T4, bản gốc Q5: code 9,8 -> 14,9 token/giây; đầu MTP nằm sẵn trong file chính chỉ được 11,6.
MTP_FILE = ("unsloth/Qwen3.8-27B-GGUF", "MTP/mtp-Qwen3.8-27B-Q4_0.gguf", 1.37)
AUTO_MAX = "Q4_K_M"      # "auto" không chọn mức nặng hơn mức này. Đo 13/10 trên 2×T4: Q4 30 token/giây, Q5 25 (code)

# --- tự chọn mức nén và ngữ cảnh theo VRAM ---
_m = MODELS[MODEL]
ALIAS = _m["alias"]
GIB = 1e9 / 2**30
# Qwen3.8-27B chỉ có 16/64 lớp full attention giữ KV: f16 tốn 64 KiB mỗi token; q8_0 và q4_0 nhỏ hơn theo số bit.
KV_KIB = {"f16": 64.0, "q8_0": 34.0, "q4_0": 18.0}
VRAM_DUNG_DUOC = sum(g["gib"] - 0.75 for g in GPUS)      # chừa khoảng 0,75 GiB mỗi card cho CUDA


def can_gib(quant, ctx, kv):
    """Ước lượng tổng VRAM cần: trọng số + KV + bộ đệm tính toán + MTP (đo 13/10 ở ngữ cảnh 32K: khoảng 2,1 GiB)
    + phần đọc ảnh. Phần đọc ảnh (file mmproj + bộ đệm khi gặp ảnh, ước 1 GiB) nằm hết trên GPU0 nên tính hai lần:
    13/10 Swift Q5 ngữ cảnh 131K chia "tensor" dùng 14.717/15.360 MiB ở GPU0 và chết ở lượt đọc ảnh đầu tiên."""
    anh = (MMPROJ_GB * GIB + 1.0) * (2 if len(GPUS) > 1 else 1) if VISION else 0
    return _m["quants"][quant] * GIB + ctx * KV_KIB[kv] / 2**20 + 1.5 + (2.1 if MTP else 0) + anh


def chon_cau_hinh():
    kv = "q8_0" if KV == "auto" else KV
    theo_co = sorted(_m["quants"], key=_m["quants"].get, reverse=True)
    # Ưu tiên ngữ cảnh rộng nhất còn vừa, trong đó lấy mức nén nặng nhất (không quá AUTO_MAX).
    for ctx in ([int(CTX)] if CTX != "auto" else (262144, 131072, 65536, 32768, 16384)):     # 262144 = tối đa của model
        for q in ([QUANT] if QUANT != "auto" else [q for q in theo_co if _m["quants"][q] <= _m["quants"][AUTO_MAX]]):
            if can_gib(q, ctx, kv) <= VRAM_DUNG_DUOC:
                return q, ctx, kv, True
    # Không nằm trọn trong VRAM: lấy mức đã chọn (hoặc nhẹ nhất), để llama.cpp tự đẩy phần thừa sang CPU (chậm).
    return (QUANT if QUANT != "auto" else theo_co[-1]), (int(CTX) if CTX != "auto" else 16384), kv, False


QUANT_CHON, CTX_SIZE, KV_TYPE, VUA_TRON = chon_cau_hinh()
MODEL_FILE = _m["file"].format(q=QUANT_CHON)
MODEL_GB = _m["quants"][QUANT_CHON]
MODEL_REPO, MMPROJ = _m["repo"], (_m["mmproj"], MMPROJ_GB)       # ô 5–6 dùng chung đọc các biến này
LLAMA_MAU = ["--temp", "1.0", "--top-p", "0.95", "--top-k", "20"]     # mặc định theo model card (chế độ có nghĩ)
print(json.dumps({
    "model": f"{_m['repo']} / {MODEL_FILE}", "alias": ALIAS, "co_file_gb": MODEL_GB,
    "ngu_canh": CTX_SIZE, "kv": KV_TYPE, "doc_anh": VISION, "nghi_mac_dinh": THINKING, "mtp": MTP_N if MTP else 0,
    "vram_dung_duoc_gib": round(VRAM_DUNG_DUOC, 1), "vram_uoc_can_gib": round(can_gib(QUANT_CHON, CTX_SIZE, KV_TYPE), 1),
    "nam_tron_trong_gpu": VUA_TRON,
}, ensure_ascii=False, indent=1))
if not VUA_TRON:
    print("CẢNH BÁO: model không nằm trọn trong VRAM, một phần chạy bằng CPU nên sẽ chậm. "
          "Kaggle: chọn GPU T4 x2. Colab Pro: chọn A100 hoặc G4.")

In [ ]:
# 3) Khóa bí mật (không bắt buộc), thư viện cho gateway, hàm dùng chung
def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name) or None
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name) or None
    except Exception:
        return None
    return None


HF_TOKEN = get_secret("HF_TOKEN")
GW_TOKEN_CO_DINH = get_secret("GATEWAY_TOKEN")
CF_TOKEN = get_secret("CF_TUNNEL_TOKEN")
PUBLIC_BASE_URL = (get_secret("PUBLIC_BASE_URL") or "").rstrip("/")
API_KEY = "sk-" + secrets.token_hex(24)          # khóa nội bộ gateway -> máy chạy model, không ra ngoài
print("HF_TOKEN:", "có" if HF_TOKEN else "không (đủ cho repo công khai)")
print("GATEWAY_TOKEN:", "lấy từ Secret (cố định)" if GW_TOKEN_CO_DINH else "không có, sẽ sinh khóa mới cho phiên này")
print("CF_TUNNEL_TOKEN:", "có (tunnel có tên)" if CF_TOKEN else "không có (Quick Tunnel, địa chỉ đổi mỗi phiên)")

_thieu = [m for m in ("fastapi", "uvicorn", "httpx", "requests", "huggingface_hub") if importlib.util.find_spec(m) is None]
if _thieu:
    print("Cài thêm:", _thieu)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.116,<1", "uvicorn>=0.35,<1",
                    "httpx>=0.28,<1", "requests>=2.31", "huggingface_hub>=0.28"], check=True)
import requests

KHONG_NGHI = {"chat_template_kwargs": {"enable_thinking": False}}


def tai(url, dst, sha256=None):
    """Tải url về dst (ghi file tạm rồi đổi tên); có sha256 thì kiểm."""
    dst = Path(dst)
    tmp = dst.with_name(dst.name + ".part")
    h, n, t0 = hashlib.sha256(), 0, time.time()
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=120) as r, \
            open(tmp, "wb") as f:
        while True:
            b = r.read(4 << 20)
            if not b:
                break
            f.write(b)
            h.update(b)
            n += len(b)
    if sha256 and h.hexdigest() != sha256:
        tmp.unlink()
        raise RuntimeError(f"Sai mã băm, không dùng file này: {dst.name}")
    tmp.replace(dst)
    print(f"  tải {dst.name}: {n / 2**20:.0f} MiB trong {time.time() - t0:.0f} giây"
          + (", sha256 khớp" if sha256 else ", KHÔNG có mã băm để kiểm"))
    return dst


def chay_nen(lenh, **kw):
    """subprocess.Popen cho tiến trình chạy lâu (máy chạy model, gateway, tunnel). Nhóm tiến trình riêng: bấm Stop một ô thì
    Jupyter gửi SIGINT cho cả nhóm của kernel (13/10 Colab G4: Stop ô 10 tắt luôn vLLM). PR_SET_PDEATHSIG: kernel chết
    (Restart session) thì tiến trình nhận SIGTERM, không sót lại giữ GPU và RAM."""
    import ctypes, signal
    prctl, sigterm = ctypes.CDLL("libc.so.6", use_errno=True).prctl, int(signal.SIGTERM)   # nạp sẵn ở tiến trình cha:
                                                                                          # trong con chỉ gọi hàm, không nạp gì

    def _theo_kernel():
        prctl(1, sigterm)                                                     # 1 = PR_SET_PDEATHSIG

    return subprocess.Popen(lenh, start_new_session=True, preexec_fn=_theo_kernel, **kw)


def hoi(messages, url=None, headers=None, timeout=900, **kw):
    """Một lượt chat không stream. Mặc định gọi thẳng máy chạy model trong máy (không qua gateway)."""
    r = requests.post(url or f"http://127.0.0.1:{ENGINE_PORT}/v1/chat/completions",
                      headers=headers or {"Authorization": f"Bearer {API_KEY}"},
                      json={"model": ALIAS, "messages": messages, **kw}, timeout=timeout)
    r.raise_for_status()
    return r.json()


def anh_7391():
    """Ảnh PNG vẽ số 7391, dạng data URL, để thử phần đọc ảnh."""
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new("RGB", (448, 224), "white")
    try:
        font = ImageFont.load_default(size=110)
    except TypeError:
        font = ImageFont.load_default()
    ImageDraw.Draw(img).text((70, 50), "7391", fill="black", font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def hoi_anh(timeout=900):
    """Hỏi số trong ảnh 7391 qua API nội bộ, trả chữ model đáp."""
    j = hoi([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": anh_7391()}},
        {"type": "text", "text": "Trong ảnh có số nào? Chỉ trả lời bằng chữ số."}]}],
        timeout=timeout, max_tokens=32, temperature=0, **KHONG_NGHI)
    return (j["choices"][0]["message"].get("content") or "").strip()

In [ ]:
# 4) llama.cpp: tải bản dựng sẵn chính thức (CUDA), kiểm mã băm, kiểm nó thấy đủ GPU. Không biên dịch.
LLAMA_URL = "https://github.com/ggml-org/llama.cpp/releases/download"
LLAMA_SHA256 = {   # trường digest của API GitHub, đọc ngày 13/10/2026. Đổi LLAMA_TAG sang bản khác thì không có mã để kiểm.
    "llama-b11429-bin-ubuntu-cuda-12.8-x64.tar.gz": "b13c64b9224d3c89d9993945d70cc21177ad00178c21b8d34237e3bc139834f1",
    "cudart-llama-b11429-bin-ubuntu-cuda-12.8-x64.tar.gz": "f4ab593e31507320b121f2cb21791f4c72420a2483d3df8bf2ca1a97d04e6845",
    "llama-b11429-bin-ubuntu-cuda-13.4-x64.tar.gz": "8082b7eaa74a714c9fecca19128f751c8e32da763ee8096b8ad1e824da7621d3",
    "cudart-llama-b11429-bin-ubuntu-cuda-13.4-x64.tar.gz": "93d18648d815b2bd624d83d82f653e1db97afb478f02064305fe3cf570040a6d",
}


def giai_nen(tar_path, dest):
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path) as t:
        t.extractall(dest, filter="data") if sys.version_info >= (3, 12) else t.extractall(dest)
    Path(tar_path).unlink()


def thu_muc_cuda():
    ds = ["/usr/local/cuda/lib64", "/usr/local/cuda/targets/x86_64-linux/lib", "/usr/local/nvidia/lib64", "/usr/lib64-nvidia"]
    for sp in site.getsitepackages():
        ds += [str(p) for p in Path(sp, "nvidia").glob("*/lib")]
    return [d for d in ds if Path(d).is_dir()]


def moi_truong(goc):
    """Biến môi trường để chạy llama-server: thêm mọi thư mục có .so dưới `goc` và thư viện CUDA của máy."""
    so_dirs = sorted({str(p.parent) for p in Path(goc).rglob("*.so*")})
    env = dict(os.environ)
    env["LD_LIBRARY_PATH"] = ":".join(d for d in [*so_dirs, *thu_muc_cuda(), env.get("LD_LIBRARY_PATH", "")] if d)
    # Bản dựng sẵn b11429 (CUDA 12.8) chỉ có mã máy cho sm_86, sm_89, sm_120a; T4 (sm_75) và A100 (sm_80) dịch từ PTX
    # lúc chạy (đo 13/10: lượt hỏi đầu mất khoảng 1 phút). Nới bộ đệm dịch để lần nạp lại trong phiên không dịch lại.
    env.setdefault("CUDA_CACHE_MAXSIZE", str(4 << 30))
    return env


def thu(server, goc):
    """Trả (số thiết bị CUDA mà llama-server thấy, chữ nó in ra)."""
    try:
        v = subprocess.run([str(server), "--version"], text=True, capture_output=True, timeout=120, env=moi_truong(goc))
        d = subprocess.run([str(server), "--list-devices"], text=True, capture_output=True, timeout=120, env=moi_truong(goc))
    except Exception as e:
        return 0, repr(e)
    out = (d.stdout or "") + (d.stderr or "")
    return len(re.findall(r"^\s*CUDA\d+:", out, re.M)), ((v.stdout or "") + (v.stderr or "")).strip() + "\n" + out.strip()


def co_cudart(major):
    return any(list(Path(d).glob(f"libcudart.so.{major}*")) for d in thu_muc_cuda())


t0 = time.time()
server_bin = LLAMA_DIR = None
# Kaggle đo 13/10/2026 dùng CUDA 12.8; Colab dùng CUDA 13.0. Thử bản khớp thư viện có sẵn trước.
for cuda in (["12.8", "13.4"] if (co_cudart(12) or not co_cudart(13)) else ["13.4", "12.8"]):
    goc = BIN_DIR / f"{LLAMA_TAG}-cuda{cuda}"
    if not any(goc.rglob("llama-server")):
        ten = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-{cuda}-x64.tar.gz"
        giai_nen(tai(f"{LLAMA_URL}/{LLAMA_TAG}/{ten}", SCRATCH / ten, LLAMA_SHA256.get(ten)), goc)
    sv = next(iter(goc.rglob("llama-server")), None)
    if sv is None:
        continue
    os.chmod(sv, 0o755)
    n_cuda, chu = thu(sv, goc)
    if n_cuda == 0:      # máy thiếu thư viện CUDA runtime đúng bản: lấy thêm gói cudart chính thức của cùng bản
        print(f"  bản CUDA {cuda} chưa thấy GPU, tải thêm thư viện CUDA đi kèm...")
        ten = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-{cuda}-x64.tar.gz"
        giai_nen(tai(f"{LLAMA_URL}/{LLAMA_TAG}/{ten}", SCRATCH / ten, LLAMA_SHA256.get(ten)), goc / "cudart")
        n_cuda, chu = thu(sv, goc)
    print(f"llama.cpp {LLAMA_TAG} (CUDA {cuda}): thấy {n_cuda}/{len(GPUS)} GPU")
    if n_cuda >= len(GPUS):
        server_bin, LLAMA_DIR = sv, goc
        break
if server_bin is None:
    raise SystemExit("llama-server dựng sẵn không thấy GPU trên máy này. Chữ nó in ra:\n" + chu[-2000:] +
                     "\nThử đổi LLAMA_TAG ở ô 2 sang bản mới hơn (https://github.com/ggml-org/llama.cpp/releases).")
LLAMA_ENV = moi_truong(LLAMA_DIR)
LLAMA_HELP = subprocess.run([str(server_bin), "--help"], text=True, capture_output=True, env=LLAMA_ENV).stdout
print(chu.strip()[:700])
print(f"[OK] llama-server sẵn sàng sau {time.time() - t0:.0f} giây: {server_bin}")

In [ ]:
# 5) Model: dùng file đã gắn ở /kaggle/input nếu có, không thì tải từ Hugging Face về SCRATCH
def lay_file(ten, gb, repo=None):
    inp = Path("/kaggle/input")
    co_san = next((p for p in inp.rglob(Path(ten).name) if p.is_file()), None) if inp.exists() else None
    if co_san:
        print(f"  dùng file đã gắn: {co_san} ({co_san.stat().st_size / 1e9:.2f} GB)")
        return co_san
    dst = MODEL_DIR / ten
    if dst.exists() and dst.stat().st_size >= gb * 1e9 * 0.98:
        print(f"  đã có: {dst}")
        return dst
    trong = shutil.disk_usage(MODEL_DIR).free / 1e9
    if trong < gb + 2:
        raise SystemExit(f"Đĩa {MODEL_DIR} còn {trong:.1f} GB, không đủ cho file {gb} GB. Chọn QUANT nhẹ hơn ở ô 2.")
    if CHAY_NEN:
        os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"      # chạy nền thì thanh tiến độ chỉ làm rối log
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    p = Path(hf_hub_download(repo or MODEL_REPO, ten, local_dir=str(MODEL_DIR), token=HF_TOKEN or None))
    dt = time.time() - t0
    print(f"  tải {ten}: {p.stat().st_size / 1e9:.2f} GB trong {dt:.0f} giây ({p.stat().st_size / 1e6 / max(dt, 1):.0f} MB/giây)")
    return p


model_path = lay_file(MODEL_FILE, MODEL_GB)
mmproj_path = lay_file(*MMPROJ) if VISION and MMPROJ else None
mtp_path = lay_file(MTP_FILE[1], MTP_FILE[2], repo=MTP_FILE[0]) if MTP and MTP_FILE else None
print("[OK] Model:", model_path)

In [ ]:
# 6) Chạy llama-server (chỉ nghe 127.0.0.1) và chờ nạp xong model
def co_co(flag):
    """llama-server bản này có nhận cờ `flag` không (đọc từ --help, để không chết khi cờ đổi tên)."""
    return re.search(rf"(?<![\w-]){re.escape(flag)}(?![\w-])", LLAMA_HELP) is not None


def cho_song(port, proc, timeout_s):
    t0, cuoi = time.time(), None
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return False, f"tiến trình thoát với mã {proc.returncode}"
        try:
            r = requests.get(f"http://127.0.0.1:{port}/health", timeout=3)
            cuoi = (r.status_code, r.text[:120])
            if r.status_code == 200:
                return True, cuoi
        except Exception as e:
            cuoi = repr(e)[:160]
        time.sleep(3)
    return False, cuoi


if globals().get("ENGINE_PROC") is not None and ENGINE_PROC.poll() is None:     # chạy lại ô này: tắt bản cũ trước
    ENGINE_PROC.terminate()
    ENGINE_PROC.wait(timeout=60)

args = [str(server_bin), "-m", str(model_path), "--alias", ALIAS,
        "--host", "127.0.0.1", "--port", str(ENGINE_PORT), "--api-key", API_KEY,
        "-np", "1", "-t", str(os.cpu_count() or 4),
        "-fa", "on", "-ctk", KV_TYPE, "-ctv", KV_TYPE, "--jinja", *LLAMA_MAU]
if VUA_TRON:
    args += ["-ngl", "99"]
if mmproj_path:
    args += ["--mmproj", str(mmproj_path)]
    # Model họ Qwen đọc sai chữ nhỏ khi ảnh ra quá ít token (llama.cpp tự cảnh báo "require at minimum 1024 image tokens").
    # Đo 13/10 với FableCoder: không cờ thì ảnh số 448×224 chỉ ra khoảng 100 token, đọc 7391 thành '71' (đúng 3/6 ảnh thử);
    # 256, 512, 1024 đều đúng 6/6. Lấy 1024 theo llama.cpp: mỗi ảnh nhỏ tốn khoảng 1.060 token ngữ cảnh, VRAM không tăng.
    _amin = globals().get("ANH_MIN_TOKEN", 1024)
    if _amin and co_co("--image-min-tokens"):
        args += ["--image-min-tokens", str(_amin)]
if not THINKING and co_co("--reasoning-budget"):
    args += ["--reasoning-budget", "0"]
# MTP_FILE None: lớp MTP nằm sẵn trong file model (không cần -md)
MTP_DUNG = bool(MTP and (mtp_path or not MTP_FILE) and co_co("--spec-type") and "draft-mtp" in LLAMA_HELP)
if MTP_DUNG:
    args += ["--spec-type", "draft-mtp", "--spec-draft-n-max", str(MTP_N)] + (["-md", str(mtp_path)] if mtp_path else []) + ["-ngld", "99"]
elif MTP:
    print("  (bản llama.cpp này không có --spec-type draft-mtp: chạy không MTP)")
args += next(([f] for f in ("--no-webui", "--no-ui") if co_co(f)), [])
args += globals().get("LLAMA_THEM", [])
# Chia model cho nhiều GPU. "tensor": mỗi lớp chia cho mọi GPU, các GPU chạy cùng lúc (llama.cpp ghi EXPERIMENTAL).
# Đo 13/10 trên 2×T4, Qwen3.8-27B gốc: "layer" 9,8–10,8 token/giây; "tensor" 12,6–13,1; "tensor" + MTP: Q5 15,7–25,3, Q4 19,1–30,0.
# "row" không dùng: bản dựng sẵn báo "device CUDA0 does not support split buffers".
# Nạp hỏng (thường là hết VRAM) thì thử lại "tensor" với nửa ngữ cảnh, rồi mới lùi về "layer".
if not VUA_TRON:
    lan_thu = [(None, CTX_SIZE)]
elif len(GPUS) < 2:      # một GPU (Colab): hỏng thì thử nửa ngữ cảnh, rồi một phần tư
    lan_thu = [(None, CTX_SIZE)] + [(None, CTX_SIZE // k) for k in (2, 4) if CTX_SIZE // k >= 16384]
elif SPLIT == "auto" and re.search(r"--split-mode[^\n]*tensor", LLAMA_HELP):
    lan_thu = [("tensor", CTX_SIZE)] + ([("tensor", CTX_SIZE // 2)] if CTX_SIZE > 32768 else []) + [("layer", CTX_SIZE)]
else:
    lan_thu = [("layer" if SPLIT == "auto" else SPLIT, CTX_SIZE)]

ENGINE_TEN, ENGINE_LOG = f"llama-server {LLAMA_TAG}", LOG_DIR / "llama-server.log"
DOC_ANH_DAU = None
for SPLIT_DUNG, _ctx in lan_thu:
    lenh = args + ["-c", str(_ctx)] + (["-sm", SPLIT_DUNG, "-ts", ",".join("1" for _ in GPUS)] if SPLIT_DUNG else [])
    print("Lệnh:", " ".join(a if a != API_KEY else "<khóa nội bộ>" for a in lenh))
    SERVER_LOG_F = open(ENGINE_LOG, "w")
    ENGINE_PROC = chay_nen(lenh, stdout=SERVER_LOG_F, stderr=subprocess.STDOUT, env=LLAMA_ENV)
    t0 = time.time()
    ok, cuoi = cho_song(ENGINE_PORT, ENGINE_PROC, timeout_s=1200)
    if ok and mmproj_path:
        # Phần đọc ảnh xin thêm bộ đệm ở GPU0 khi gặp ảnh đầu tiên; 13/10 server chết đúng lúc đó vì GPU0 hết VRAM.
        # Hỏi thử một ảnh ngay đây: server chết thì coi như nạp hỏng và thử cấu hình kế tiếp.
        try:
            DOC_ANH_DAU = hoi_anh()
        except Exception as e:
            DOC_ANH_DAU = f"LỖI {type(e).__name__}: {str(e)[:160]}"
        if ENGINE_PROC.poll() is not None:
            ok, cuoi = False, "server chết ở lượt đọc ảnh đầu tiên (thường là GPU0 hết VRAM)"
    if ok:
        CTX_SIZE = _ctx
        break
    if ENGINE_PROC.poll() is None:
        ENGINE_PROC.kill()
    SERVER_LOG_F.close()
    print(f"  không nạp được (chia {SPLIT_DUNG}, ngữ cảnh {_ctx}): {cuoi}. Cuối log:\n" + ENGINE_LOG.read_text(errors="ignore")[-600:])
if not ok:
    raise SystemExit(f"llama-server không lên ({cuoi}). Cuối log:\n" + ENGINE_LOG.read_text(errors="ignore")[-4000:] +
                     "\nNếu là hết bộ nhớ (out of memory): hạ CTX hoặc chọn QUANT nhẹ hơn ở ô 2, hoặc đặt MTP = False.")
print(f"[OK] Model nạp xong sau {time.time() - t0:.0f} giây (PID {ENGINE_PROC.pid}, chia GPU: {SPLIT_DUNG or 'không'}, "
      f"ngữ cảnh {CTX_SIZE})" + (f"; đọc ảnh lần đầu: {DOC_ANH_DAU!r}" if DOC_ANH_DAU is not None else ""))
_log = ENGINE_LOG.read_text(errors="ignore")
for _dong in re.findall(r"[^\n]*(?:model buffer size|KV buffer size|compute buffer size)[^\n]*", _log)[:8]:
    print("  ", _dong.strip()[:160])
for i, used, total in gpu_csv("index,memory.used,memory.total"):
    print(f"   GPU {i}: {used}/{total} MiB")
DOC_ANH, MTP_TOKENS = mmproj_path is not None, (MTP_N if MTP_DUNG else 0)     # các ô dùng chung 7–10 đọc hai biến này

In [ ]:
# 7) Tự kiểm qua API: trả lời, tốc độ, nghĩ + stream, gọi tool, đọc ảnh. Lỗi ở hai mục đầu thì dừng.
LOCAL = f"http://127.0.0.1:{ENGINE_PORT}/v1/chat/completions"
H_LOCAL = {"Authorization": f"Bearer {API_KEY}"}
KET_QUA = {}


def do_toc_do(de, max_tokens=300):
    """Đo tốc độ sinh của một lượt stream (token/giây, không tính đoạn đọc đề). Trả (số, chữ mô tả).
    llama-server tự đo (trường timings) thì lấy số đó; máy khác (vLLM) thì tính ở đây:
    (số token - 1) / (giờ nhận token cuối - giờ nhận token đầu)."""
    t_dau = t_cuoi = None
    n_khuc, usage, tm = 0, {}, {}
    mtp0 = DEM_MTP() if globals().get("DEM_MTP") else None
    with requests.post(LOCAL, headers=H_LOCAL, stream=True, timeout=900, json={
            "model": ALIAS, "stream": True, "stream_options": {"include_usage": True}, "max_tokens": max_tokens,
            "temperature": 0.7, "messages": [{"role": "user", "content": de}], **KHONG_NGHI}) as s:
        s.raise_for_status()
        for line in s.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:") or "[DONE]" in line:
                continue
            try:
                c = json.loads(line[5:])
            except Exception:
                continue
            usage, tm = c.get("usage") or usage, c.get("timings") or tm
            if any((ch.get("delta") or {}).get("content") for ch in c.get("choices") or []):
                t_cuoi = time.time()
                t_dau = t_dau or t_cuoi
                n_khuc += 1
    n = usage.get("completion_tokens") or n_khuc
    tinh = (n - 1) / (t_cuoi - t_dau) if t_dau and t_cuoi > t_dau and n > 1 else None
    so = tm.get("predicted_per_second") or tinh
    trung, ten_doan = "", globals().get("SPEC_TEN") or "MTP"     # SPEC_TEN: ô 6 vLLM đặt (DSpark, DFlash2, MTP)
    if tm.get("draft_n"):
        trung = f", {ten_doan} đoán trúng {tm['draft_n_accepted']}/{tm['draft_n']}"
    elif mtp0 is not None:
        mtp1 = DEM_MTP()
        if mtp1 and mtp1[1] > mtp0[1]:
            trung = f", {ten_doan} đoán trúng {mtp1[0] - mtp0[0]:.0f}/{mtp1[1] - mtp0[1]:.0f}"
    return so, (f"{so:.1f} token/giây ({n} token{trung})" if so else "không đo được")


# (a) trả lời đúng một chuỗi, không nghĩ
j = hoi([{"role": "user", "content": "Reply with exactly: QWEN_OK"}], max_tokens=32, temperature=0, **KHONG_NGHI)
_txt = (j["choices"][0]["message"].get("content") or "").strip()
assert "QWEN_OK" in _txt, f"Trả lời sai: {_txt!r} | {json.dumps(j)[:400]}"
KET_QUA["trả lời (không nghĩ)"] = f"đạt: {_txt!r}"

# (b) tốc độ sinh chữ: một đoạn code và một đoạn văn tiếng Việt. Lượt sinh dài đầu tiên sau khi nạp còn gánh phần khởi động
# của đường đoán trước (13/10 FableCoder MTP 2: lượt code đo đầu tiên 27,6 token/giây, bài đo có làm nóng 93,0): chạy một lượt nháp.
# Có lượt nháp rồi vẫn có lúc lượt code đầu bị khựng (13/10 13:46: 48,1 trong khi văn đo liền sau 75,0, các lượt khác 91–94):
# đo code hai lần, lấy lượt nhanh hơn, và in cả lượt đầu khi nó chậm hơn từ 20%.
do_toc_do("Đếm từ 1 đến 40, mỗi số một dòng.", max_tokens=96)
_hai_lan = [do_toc_do("Viết lớp Python HangDoi (hàng đợi ưu tiên dùng heapq) có các hàm them, lay, xem, kèm docstring. Chỉ trả code.")
            for _ in range(2)]
TOC_DO, _code = max(_hai_lan, key=lambda x: x[0] or 0)
if _hai_lan[0][0] and TOC_DO and _hai_lan[0][0] < 0.8 * TOC_DO:
    _code += f", lượt đầu chỉ {_hai_lan[0][0]:.1f}"
_, _van = do_toc_do("Viết một đoạn văn khoảng 200 từ giới thiệu Việt Nam.")
KET_QUA["tốc độ sinh"] = f"code {_code} | văn {_van}"

# (c) có nghĩ + stream, mức nghĩ "low". Bật nghĩ rõ ràng:
# model Qwen nhỏ mặc định không nghĩ (13/10 bài thử Qwen3.5-2B trả "85", phần nghĩ 0 ký tự)
try:
    _nghi, _noi = [], []
    with requests.post(LOCAL, headers=H_LOCAL, stream=True, timeout=900, json={
            "model": ALIAS, "stream": True, "max_tokens": 4096,
            "messages": [{"role": "user", "content": "17 nhân 23 bằng bao nhiêu? Trả lời một con số."}],
            "chat_template_kwargs": {"enable_thinking": True, "reasoning_effort": "low"}}) as s:
        s.raise_for_status()
        for line in s.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:") or "[DONE]" in line:
                continue
            try:
                d = (json.loads(line[5:]).get("choices") or [{}])[0].get("delta", {})
            except Exception:
                continue
            _nghi.append(d.get("reasoning_content") or d.get("reasoning") or "")
            _noi.append(d.get("content") or "")
    _nghi, _noi = "".join(_nghi), "".join(_noi)
    KET_QUA["nghĩ + stream"] = (f"đạt: trả {_noi.strip()[:40]!r}, phần nghĩ {len(_nghi)} ký tự tách riêng" if "391" in _noi
                                else f"SAI: trả {_noi.strip()[:80]!r} (đúng là 391), phần nghĩ {len(_nghi)} ký tự")
except Exception as e:
    KET_QUA["nghĩ + stream"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"

# (d) gọi tool
try:
    j = hoi([{"role": "user", "content": "Thời tiết ở Hà Nội bây giờ thế nào? Hãy gọi công cụ."}], max_tokens=512, **KHONG_NGHI,
            tools=[{"type": "function", "function": {
                "name": "get_weather", "description": "Lấy thời tiết hiện tại của một thành phố",
                "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}])
    _tc = j["choices"][0]["message"].get("tool_calls") or []
    KET_QUA["gọi tool"] = (f"đạt: {_tc[0]['function']['name']}({_tc[0]['function']['arguments']})" if _tc
                           else f"KHÔNG gọi tool, trả chữ: {(j['choices'][0]['message'].get('content') or '')[:80]!r}")
except Exception as e:
    KET_QUA["gọi tool"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"

# (e) đọc ảnh: ảnh vẽ số 7391
if DOC_ANH:
    try:
        _txt = hoi_anh()
        KET_QUA["đọc ảnh"] = f"đạt: đọc ra {_txt!r}" if "7391" in _txt else f"SAI: đọc ra {_txt!r} (đúng là 7391)"
    except Exception as e:
        KET_QUA["đọc ảnh"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"
else:
    KET_QUA["đọc ảnh"] = "tắt (VISION = False)"

for k, v in KET_QUA.items():
    print(f"  {k:<22} {v}")
if ENGINE_PROC.poll() is not None:
    raise SystemExit(f"{ENGINE_TEN} đã chết trong lúc tự kiểm (mã {ENGINE_PROC.returncode}). Cuối log:\n"
                     + ENGINE_LOG.read_text(errors="ignore")[-3000:]
                     + "\nThấy 'out of memory': hạ CTX (ví dụ 65536) hoặc đặt VISION = False ở ô 2, rồi chạy lại từ ô 2.")
print("[OK] Model trả lời đúng qua API nội bộ.")

In [ ]:
%%writefile gateway.py
"""Narrow OpenAI-compatible inference gateway. Tunnel THIS port (8787), never llama-server (8080) directly.

Security Invariants:
1. Strict allowlist: ONLY /healthz, /capabilities, /v1/models, /v1/chat/completions.
2. Distinct public GATEWAY_TOKEN vs internal VLLM_API_KEY.
3. Unknown routes return 404 (no catch-all proxy).
4. Unbuffered SSE streaming pass-through.
5. Strict request body limit (25 MB default).
6. Personal single-client concurrency guard (semaphore).
7. Zero internal credential or traceback leakage.
8. Request ID correlation logging without prompt content exposure.
9. Clean async lifespan management and shutdown.
"""
import asyncio
from contextlib import asynccontextmanager
import json
import logging
import os
import secrets
import time
from typing import AsyncGenerator

from fastapi import FastAPI, Header, HTTPException, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
import httpx

# Logging configuration (Request ID and metadata only, NO prompt/body contents)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("qwen_gateway")

# Configuration & secrets
VLLM_URL = os.getenv("VLLM_URL", "http://127.0.0.1:8080").rstrip("/")
GATEWAY_TOKEN = os.getenv("GATEWAY_TOKEN", "")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "")
MODEL = os.getenv("SERVED_MODEL_NAME", "qwen3.8-27b")
MAX_MODEL_LEN = int(os.getenv("MAX_MODEL_LEN", "131072"))
MULTIMODAL = os.getenv("TEXT_ONLY", "false").lower() not in {"1", "true", "yes"}
MTP_TOKENS = int(os.getenv("MTP_TOKENS", "3"))
RUN_ID = os.getenv("RUN_ID", str(int(time.time())))
MAX_BODY_BYTES = int(os.getenv("GATEWAY_MAX_BODY_BYTES", str(25 * 1024 * 1024))) # 25 MB
MAX_CONCURRENT_INFERENCE = int(os.getenv("GATEWAY_CONCURRENCY", "3"))
MAX_OUTPUT_TOKENS = int(os.getenv("GATEWAY_MAX_OUTPUT_TOKENS", "32768"))
NONSTREAM_WAIT_S = float(os.getenv("GATEWAY_NONSTREAM_WAIT_S", "60"))  # below Cloudflare's 100 s cut-off
NONSTREAM_PAD_S = 15.0

if not GATEWAY_TOKEN or not VLLM_API_KEY:
    raise RuntimeError("Both GATEWAY_TOKEN and VLLM_API_KEY environment variables are required")

# State & Lifespan
client_holder: dict[str, httpx.AsyncClient] = {}
concurrency_limit = asyncio.Semaphore(MAX_CONCURRENT_INFERENCE)

@asynccontextmanager
async def lifespan(app: FastAPI):
    limits = httpx.Limits(max_keepalive_connections=10, max_connections=20)
    timeout = httpx.Timeout(900.0, connect=10.0, read=900.0, write=30.0)
    client = httpx.AsyncClient(limits=limits, timeout=timeout)
    client_holder["client"] = client
    logger.info(f"Gateway initialized. Target upstream at {VLLM_URL}, Model: {MODEL}")
    try:
        yield
    finally:
        logger.info("Gateway shutting down, closing upstream connections...")
        await client.aclose()
        logger.info("Gateway shutdown complete.")

app = FastAPI(
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
    lifespan=lifespan,
)

# CORS for browser clients (`npm run dev` on localhost calling the tunnel URL).
# Auth is NOT weakened: preflight OPTIONS carries no credentials, and every
# allowlisted route still requires the Bearer gateway token.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"],
    max_age=600,
)

@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    logger.error(f"Unhandled error handling {request.method} {request.url.path}: {type(exc).__name__}")
    return JSONResponse(
        status_code=500,
        content={"error": {"message": "Internal gateway error", "type": "gateway_error", "code": 500}},
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": {"message": exc.detail, "type": "http_error", "code": exc.status_code}},
    )

def auth(authorization: str | None) -> None:
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing or invalid authorization header")
    provided_token = authorization[7:].strip()
    if not secrets.compare_digest(provided_token, GATEWAY_TOKEN):
        raise HTTPException(401, "Unauthorized: invalid gateway token")

def internal_headers() -> dict[str, str]:
    return {
        "Authorization": f"Bearer {VLLM_API_KEY}",
        "Content-Type": "application/json",
    }

async def read_bounded_body(request: Request) -> bytes:
    content_length = request.headers.get("content-length")
    if content_length:
        try:
            declared = int(content_length)
        except ValueError:
            raise HTTPException(400, "Malformed Content-Length header")
        if declared > MAX_BODY_BYTES:
            raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")

    body = bytearray()
    async for chunk in request.stream():
        body.extend(chunk)
        if len(body) > MAX_BODY_BYTES:
            raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")
    return bytes(body)

@app.get("/healthz")
async def healthz(authorization: str | None = Header(default=None)):
    auth(authorization)
    req_id = secrets.token_hex(4)
    client = client_holder.get("client")
    if not client:
        return JSONResponse({"ok": False, "run_id": RUN_ID, "error": "Gateway starting"}, status_code=503)

    try:
        r = await client.get(f"{VLLM_URL}/v1/models", headers=internal_headers(), timeout=8.0)
        is_ready = r.status_code == 200
        logger.info(f"[{req_id}] /healthz -> upstream {r.status_code} (ready={is_ready})")
        return JSONResponse(
            {"ok": is_ready, "run_id": RUN_ID, "model": MODEL},
            status_code=200 if is_ready else 503,
        )
    except Exception as e:
        logger.warning(f"[{req_id}] /healthz check failed: {type(e).__name__}")
        return JSONResponse(
            {"ok": False, "run_id": RUN_ID, "error": "Upstream unavailable"},
            status_code=503,
        )

@app.get("/capabilities")
async def capabilities(authorization: str | None = Header(default=None)):
    auth(authorization)
    return {
        "runId": RUN_ID,
        "model": MODEL,
        "maxModelLen": MAX_MODEL_LEN,
        "multimodal": MULTIMODAL,
        "toolCalling": True,
        "streamUsage": True,
        "reasoningLevels": ["low", "medium", "xhigh"],
        "defaultReasoning": "xhigh",
        "backend": {
            "vllm_tpu": os.getenv("VLLM_TPU_VERSION", "0.28.0"),
            "mtp_tokens": MTP_TOKENS,
        },
    }

@app.get("/v1/models")
async def models(request: Request):
    auth(request.headers.get("authorization"))
    req_id = secrets.token_hex(4)
    client = client_holder.get("client")
    if not client:
        raise HTTPException(503, "Gateway unready")

    try:
        r = await client.get(f"{VLLM_URL}/v1/models", headers=internal_headers(), timeout=10.0)
        logger.info(f"[{req_id}] GET /v1/models -> {r.status_code}")
        try:
            content = r.json() if r.content else {}
        except Exception:
            content = {"error": {"message": f"Upstream returned non-JSON (status {r.status_code})", "type": "upstream_error", "code": r.status_code}}
        return JSONResponse(status_code=r.status_code, content=content)
    except httpx.RequestError:
        raise HTTPException(503, "Upstream inference service unavailable")

@app.post("/v1/chat/completions")
async def chat(request: Request):
    auth(request.headers.get("authorization"))
    req_id = secrets.token_hex(4)
    t0 = time.time()

    body = await read_bounded_body(request)

    try:
        payload = json.loads(body)
    except Exception:
        raise HTTPException(400, "Malformed JSON request body")
    if not isinstance(payload, dict):
        raise HTTPException(400, "Request body must be a JSON object")

    messages = payload.get("messages")
    if not isinstance(messages, list) or not messages:
        raise HTTPException(400, "'messages' must be a non-empty array")
    payload["model"] = MODEL
    
    # Handle both max_tokens and max_completion_tokens
    max_tokens = payload.get("max_tokens") or payload.get("max_completion_tokens")
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (TypeError, ValueError):
            raise HTTPException(400, "'max_tokens' must be an integer")
        clamped_tokens = max(1, min(max_tokens, MAX_OUTPUT_TOKENS))
        payload["max_tokens"] = clamped_tokens
        if "max_completion_tokens" in payload:
            payload["max_completion_tokens"] = clamped_tokens

    stream_raw = payload.get("stream", False)
    stream = (
        stream_raw is True
        or stream_raw == 1
        or (isinstance(stream_raw, str) and stream_raw.lower() in ("1", "true"))
    )

    client = client_holder.get("client")
    if not client:
        raise HTTPException(503, "Gateway unready")

    body = json.dumps(payload).encode("utf-8")
    if len(body) > MAX_BODY_BYTES:
        raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")

    logger.info(f"[{req_id}] POST /v1/chat/completions (stream={stream}, bytes={len(body)})")

    if not stream:
        try:
            await asyncio.wait_for(concurrency_limit.acquire(), timeout=30)
        except asyncio.TimeoutError:
            raise HTTPException(429, "Gateway busy: all inference slots occupied, retry shortly")

        async def call_upstream() -> tuple[int, dict]:
            try:
                r = await client.post(
                    f"{VLLM_URL}/v1/chat/completions",
                    content=body,
                    headers=internal_headers(),
                )
                logger.info(f"[{req_id}] Completed in {time.time() - t0:.2f}s (status={r.status_code})")
                try:
                    content = r.json() if r.content else {}
                except Exception:
                    content = {"error": {"message": f"Upstream returned non-JSON (status {r.status_code})", "type": "upstream_error", "code": r.status_code}}
                return r.status_code, content
            except httpx.RequestError:
                return 503, {"error": {"message": "Upstream inference service unavailable", "type": "http_error", "code": 503}}
            finally:
                concurrency_limit.release()

        task = asyncio.create_task(call_upstream())
        try:
            status, content = await asyncio.wait_for(asyncio.shield(task), timeout=NONSTREAM_WAIT_S)
            return JSONResponse(status_code=status, content=content)
        except asyncio.TimeoutError:
            pass

        # Cloudflare returns 524 when the origin sends nothing for 100 s. Past NONSTREAM_WAIT_S, send the
        # 200 header now and one space every 15 s (JSON allows leading whitespace), then the body.
        # Trade-off: an upstream error after this point arrives as an error JSON under status 200.
        logger.info(f"[{req_id}] Non-stream still running after {NONSTREAM_WAIT_S:.0f}s, keepalive padding on")

        async def padded() -> AsyncGenerator[bytes, None]:
            try:
                while not task.done():
                    done, _ = await asyncio.wait({task}, timeout=NONSTREAM_PAD_S)
                    if not done:
                        yield b" "
                yield json.dumps(task.result()[1]).encode("utf-8")
            finally:
                if not task.done():
                    task.cancel()
                    logger.info(f"[{req_id}] Downstream disconnected, aborting upstream call")

        return StreamingResponse(
            padded(),
            media_type="application/json",
            headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
        )

    async def sse_generator() -> AsyncGenerator[bytes, None]:
        try:
            try:
                await asyncio.wait_for(concurrency_limit.acquire(), timeout=30)
            except asyncio.TimeoutError:
                err_payload = json.dumps({"error": {"message": "Gateway busy: all inference slots occupied, retry shortly", "type": "gateway_busy", "code": 429}})
                yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
                return
            try:
                async with client.stream(
                    "POST",
                    f"{VLLM_URL}/v1/chat/completions",
                    content=body,
                    headers=internal_headers(),
                ) as r:
                    if r.status_code != 200:
                        error_bytes = await r.aread()
                        logger.warning(f"[{req_id}] Upstream error {r.status_code}")
                        try:
                            detail = json.loads(error_bytes.decode("utf-8")).get("error", {}).get(
                                "message", f"Upstream error {r.status_code}"
                            ) if error_bytes else f"Upstream error {r.status_code}"
                        except Exception:
                            detail = f"Upstream error {r.status_code}"
                        err_payload = json.dumps(
                            {"error": {"message": detail, "type": "upstream_error", "code": r.status_code}}
                        )
                        yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
                        return

                    async for chunk in r.aiter_bytes():
                        if await request.is_disconnected():
                            logger.info(f"[{req_id}] Downstream disconnected, aborting upstream pull")
                            break
                        yield chunk
            finally:
                concurrency_limit.release()
        except httpx.RequestError as e:
            logger.error(f"[{req_id}] Upstream connection broken during streaming: {type(e).__name__}")
            err_payload = json.dumps({"error": {"message": "Upstream connection broken", "type": "upstream_disconnect"}})
            yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
        finally:
            logger.info(f"[{req_id}] Stream closed after {time.time() - t0:.2f}s")

    return StreamingResponse(
        sse_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
            "Connection": "keep-alive",
        },
    )

In [ ]:
# 9) Gateway :8787 + tunnel -> in ENDPOINT. Chỉ gateway được đưa ra ngoài, máy chạy model thì không.
GW_TOKEN = GW_TOKEN_CO_DINH or ("gw-" + secrets.token_hex(20))
GW_FILE = Path("gateway.py").resolve()
assert GW_FILE.exists(), "Chưa chạy ô 8 (%%writefile gateway.py)."
for _ten in ("cf_proc", "gw_proc"):                                # chạy lại ô này: tắt bản cũ trước
    _p = globals().get(_ten)
    if _p is not None and _p.poll() is None:
        _p.terminate()
if ENGINE_PROC.poll() is not None:     # 13/10: bấm Stop ô 10 (bản notebook cũ) tắt luôn máy chạy model, gateway chờ vô ích
    raise SystemExit(f"{ENGINE_TEN} đã tắt (mã thoát {ENGINE_PROC.returncode}), gateway không có gì để nối. "
                     f"Chạy lại từ ô 6 (bấm vào ô 6 rồi Ctrl+F10). Log trong {LOG_DIR}")

gw_log = LOG_DIR / "gateway.log"
gw_proc = chay_nen(
    [sys.executable, "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", str(PORT_GW),
     "--workers", "1", "--log-level", "warning"],
    cwd=str(GW_FILE.parent), stdout=open(gw_log, "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ, GATEWAY_TOKEN=GW_TOKEN, VLLM_API_KEY=API_KEY, VLLM_URL=f"http://127.0.0.1:{ENGINE_PORT}",
             SERVED_MODEL_NAME=ALIAS, MAX_MODEL_LEN=str(CTX_SIZE), TEXT_ONLY="false" if DOC_ANH else "true",
             GATEWAY_MAX_OUTPUT_TOKENS="32768", MTP_TOKENS=str(MTP_TOKENS), VLLM_TPU_VERSION=ENGINE_TEN,
             # số lượt gateway cho chạy cùng lúc = số lượt vLLM chạy song song (gateway.py mặc định 3: 13/10 G4 MAX_SEQS 4,
             # lượt thứ tư chờ rồi bị trả 429 "gateway busy")
             GATEWAY_CONCURRENCY=str(globals().get("MAX_SEQS", 3)),
             RUN_ID=str(int(time.time()))))
H_GW = {"Authorization": f"Bearer {GW_TOKEN}"}
LOCAL_GW = f"http://127.0.0.1:{PORT_GW}"
for _ in range(30):
    if gw_proc.poll() is not None:
        raise SystemExit("Gateway chết:\n" + gw_log.read_text(errors="ignore")[-3000:])
    try:
        if requests.get(LOCAL_GW + "/healthz", headers=H_GW, timeout=5).json().get("ok") is True:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise SystemExit("Gateway không trả /healthz sau 30 giây:\n" + gw_log.read_text(errors="ignore")[-3000:])
print(f"gateway chạy ở 127.0.0.1:{PORT_GW} -> {ENGINE_TEN} 127.0.0.1:{ENGINE_PORT}")

MO_TUNNEL = (not ON_COLAB) if TUNNEL == "auto" else bool(TUNNEL)
BASE_URL, cf_proc = LOCAL_GW, None
if not MO_TUNNEL:
    print("KHÔNG mở tunnel: endpoint chỉ dùng được bên trong máy này.")
    if ON_COLAB:
        print('  Lý do mặc định tắt trên Colab: FAQ của Colab (đọc 13/10/2026) ghi "The following are disallowed from all managed\n'
              '  Colab runtimes", trong đó có "file hosting, media serving, or other web service offerings not related to interactive\n'
              '  compute with Colab" và "connecting to remote proxies" (https://research.google.com/colaboratory/faq.html).\n'
              "  Endpoint qua tunnel có thuộc diện đó không thì Colab không nói rõ; gói Pro không gỡ được danh sách này.\n"
              "  Anh tự quyết: đặt TUNNEL = True ở ô 2 rồi chạy lại ô 2 và ô 9.")
else:
    CF_VERSION = "2026.10.0"      # sha256 từ API GitHub ngày 13/10/2026
    CF_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"
    cf_bin = BIN_DIR / f"cloudflared-{CF_VERSION}"
    if not cf_bin.exists():
        tai(f"https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64", cf_bin, CF_SHA256)
        os.chmod(cf_bin, 0o755)
    cf_log = LOG_DIR / "cloudflared.log"
    if CF_TOKEN:                  # tunnel có tên: địa chỉ cố định, cấu hình trong bảng điều khiển Cloudflare trỏ về http://127.0.0.1:8787
        cf_proc = chay_nen([str(cf_bin), "tunnel", "--no-autoupdate", "run"], stdout=open(cf_log, "w"),
                           stderr=subprocess.STDOUT, env=dict(os.environ, TUNNEL_TOKEN=CF_TOKEN))
        BASE_URL = PUBLIC_BASE_URL or None
        if not BASE_URL:
            print("Tunnel có tên đã chạy nhưng thiếu Secret PUBLIC_BASE_URL (ví dụ https://qwen.tenmien.com): không tự kiểm qua tunnel được.")
    else:                         # Quick Tunnel: không cần tài khoản, địa chỉ đổi mỗi phiên
        cf_proc = chay_nen([str(cf_bin), "tunnel", "--no-autoupdate", "--url", LOCAL_GW], stdout=open(cf_log, "w"),
                           stderr=subprocess.STDOUT)
        BASE_URL, _han = None, time.time() + 120
        while time.time() < _han and not BASE_URL and cf_proc.poll() is None:
            _tim = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", cf_log.read_text(errors="ignore"))
            BASE_URL = _tim.group(0) if _tim else None
            time.sleep(1)
        if not BASE_URL:
            raise SystemExit("Không mở được Quick Tunnel:\n" + cf_log.read_text(errors="ignore")[-3000:])
    if BASE_URL:                  # kiểm đường thật từ ngoài vào: /healthz rồi một lượt chat
        _ua = {**H_GW, "User-Agent": "Mozilla/5.0"}
        for _lan in range(40):
            try:
                if requests.get(BASE_URL + "/healthz", headers=_ua, timeout=10).json().get("ok") is True:
                    break
            except Exception:
                pass
            time.sleep(3)
        else:
            raise SystemExit(f"Gateway chưa trả lời qua tunnel sau 2 phút: {BASE_URL}")
        for _lan_chat in range(6):    # tunnel mới mở còn chập chờn: 13/10 /healthz đạt rồi lượt chat liền sau báo "Network is unreachable"
            try:
                j = hoi([{"role": "user", "content": "Reply with exactly: TUNNEL_OK"}], url=BASE_URL + "/v1/chat/completions",
                        headers=_ua, max_tokens=32, temperature=0, **KHONG_NGHI)
                break
            except requests.RequestException as e:
                _loi_chat = f"{type(e).__name__}: {str(e)[:200]}"
                time.sleep(5)
        else:
            raise SystemExit(f"/healthz qua tunnel đạt nhưng lượt chat hỏng 6 lần liền ({_loi_chat}). Chạy lại ô này.")
        assert "TUNNEL_OK" in (j["choices"][0]["message"].get("content") or ""), f"Chat qua tunnel trả sai: {json.dumps(j)[:300]}"
        print(f"[OK] Đã gọi được từ ngoài vào qua tunnel (thử /healthz lần {_lan + 1}, một lượt chat đúng).")

_url = BASE_URL or "<địa chỉ tunnel có tên của anh>"
_key = "<giá trị Secret GATEWAY_TOKEN của anh>" if GW_TOKEN_CO_DINH else GW_TOKEN     # khóa cố định thì không in ra output
print("\n" + "=" * 78)
print("  ENDPOINT SẴN SÀNG (chuẩn OpenAI)")
print("=" * 78)
print(f"  Base URL : {_url}/v1")
print(f"  API key  : {_key}")
print(f"  Model    : {ALIAS}")
_anh = "không" if not DOC_ANH else "có" if KET_QUA.get("đọc ảnh", "").startswith("đạt") else "có, nhưng ô 7 đọc SAI ảnh thử"
print(f"  Ngữ cảnh : {CTX_SIZE} token | đọc ảnh: {_anh} | nghĩ mặc định: {'bật' if THINKING else 'tắt'}"
      + (f" | {globals().get('SPEC_TEN') or 'MTP'} {MTP_TOKENS} token" if MTP_TOKENS else "")
      + (f" | code {TOC_DO:.1f} token/giây" if TOC_DO else ""))
print("-" * 78)
print("  Thử nhanh bằng curl:")
print(f"    curl {_url}/v1/chat/completions -H \"Authorization: Bearer {_key}\" -H \"Content-Type: application/json\" \\")
print(f"      -d '{{\"model\":\"{ALIAS}\",\"stream\":true,\"messages\":[{{\"role\":\"user\",\"content\":\"Xin chào\"}}]}}'")
print("  Python:")
print(f"    from openai import OpenAI; c = OpenAI(base_url=\"{_url}/v1\", api_key=\"<API key>\")")
print("-" * 78)
print("  Đường được mở: GET /healthz | GET /capabilities | GET /v1/models | POST /v1/chat/completions")
print("  Tắt nghĩ cho một lượt : thêm \"chat_template_kwargs\": {\"enable_thinking\": false}")
print("  Chỉnh mức nghĩ        : \"chat_template_kwargs\": {\"reasoning_effort\": \"low\" | \"medium\" | \"xhigh\"}")
print("  Nên gọi với \"stream\": true. Lượt không stream quá 60 giây: gateway gửi khoảng trắng giữ kết nối\n"
      "  (Cloudflare cắt lượt im lặng 100 giây), nên lỗi xảy ra sau mốc đó về dưới mã 200 kèm JSON lỗi.")
if MO_TUNNEL and not CF_TOKEN:
    print("  Địa chỉ trycloudflare.com và API key đổi mỗi phiên. Muốn cố định: xem bảng Secret ở đầu notebook.")
print("=" * 78)

### Ô 9b: kiểm chất lượng
Tự chạy sau khi endpoint đã sẵn sàng (lúc này app đã gọi được). Hỏi qua gateway, đúng đường app gọi: 3 bài code, mỗi bài 3 lần
(máy tự chạy code model viết với bộ test), 2 bài toán có nghĩ, gọi tool hai lượt, đọc ảnh, tìm một dòng trong ngữ cảnh dài,
rồi phần làm nội dung: chép phụ đề tiếng Trung trên khung hình (OCR, máy tính tỉ lệ sai chữ), dịch Trung/Anh sang Việt đúng
xưng hô, dịch một lô phụ đề đánh số, tiêu đề, lời mở đầu, lời bình shorts, mô tả, một câu hỏi phải biết nói "không biết".
Phần chữ in ra cuối để đọc chấm. Mất khoảng 5–15 phút tùy máy. Không muốn chạy thì đặt `KIEM_CHAT_LUONG = False` ở ô 2.
Ô này hỏng ở bài nào thì chỉ ghi vào bảng, không làm dừng ô 10.

In [ ]:
# 9b) Kiểm chất lượng qua gateway
import base64, hashlib, io, json, os, random, re, subprocess, sys, tempfile, time, unicodedata
from pathlib import Path

import requests

_g = globals()
CL_URL = (_g.get("LOCAL_GW") or os.environ.get("CL_URL", "http://127.0.0.1:8787")).rstrip("/")
CL_H = _g.get("H_GW") or {"Authorization": f"Bearer {os.environ.get('CL_KEY', '')}"}
CL_MODEL = _g.get("ALIAS") or os.environ.get("CL_MODEL", "qwen3.8-27b")
CL_ANH = _g["DOC_ANH"] if "DOC_ANH" in _g else os.environ.get("CL_ANH", "1") == "1"
CL_CHAY_CODE = ("ENGINE_PROC" in _g) or os.environ.get("CL_CHAY_CODE") == "1"
CL_DAI = int(_g.get("CL_DAI") or os.environ.get("CL_DAI", "30000"))
CL_LAP = int(_g.get("CL_LAP") or os.environ.get("CL_LAP", "3"))
CJK = re.compile(r"[぀-ヿ一-鿿가-힯]")


def goi(messages, nghi=False, muc="low", max_tokens=2048, tools=None):
    """Một lượt stream qua gateway. Trả dict: noi, nghi, tool_calls, n (token sinh), giay, tok_s, tok_de, cho_dau."""
    body = {"model": CL_MODEL, "stream": True, "stream_options": {"include_usage": True}, "max_tokens": max_tokens,
            "messages": messages,
            "chat_template_kwargs": {"enable_thinking": True, "reasoning_effort": muc} if nghi else {"enable_thinking": False}}
    if tools:
        body["tools"] = tools
    noi, ng, tcs, usage = [], [], {}, {}
    t0 = time.time()
    t_dau = t_cuoi = None
    with requests.post(CL_URL + "/v1/chat/completions", headers=CL_H, json=body, stream=True, timeout=1800) as s:
        s.raise_for_status()
        for line in s.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:") or "[DONE]" in line:
                continue
            try:
                c = json.loads(line[5:])
            except Exception:
                continue
            usage = c.get("usage") or usage
            for ch in c.get("choices") or []:
                d = ch.get("delta") or {}
                for tc in d.get("tool_calls") or []:
                    o = tcs.setdefault(tc.get("index", 0), {"name": "", "arguments": "", "id": tc.get("id")})
                    f = tc.get("function") or {}
                    o["name"] += f.get("name") or ""
                    o["arguments"] += f.get("arguments") or ""
                    o["id"] = o["id"] or tc.get("id")
                manh = (d.get("content") or "") + (d.get("reasoning_content") or d.get("reasoning") or "")
                if manh or d.get("tool_calls"):
                    t_cuoi = time.time()
                    t_dau = t_dau or t_cuoi
                noi.append(d.get("content") or "")
                ng.append(d.get("reasoning_content") or d.get("reasoning") or "")
    n = usage.get("completion_tokens") or 0
    sinh = (t_cuoi - t_dau) if t_dau and t_cuoi else 0
    return {"noi": "".join(noi).strip(), "nghi": "".join(ng), "tool_calls": [tcs[k] for k in sorted(tcs)], "n": n,
            "giay": time.time() - t0, "tok_s": (n / sinh) if n and sinh > 0 else 0, "tok_de": usage.get("prompt_tokens"),
            "cho_dau": (t_dau - t0) if t_dau else None}


def lay_code(txt):
    khoi = re.findall(r"```(?:python|py)?\s*\n(.*?)```", txt, re.S)
    return max(khoi, key=len) if khoi else txt


def chay_code(code, test):
    if not CL_CHAY_CODE:
        return None, "không chạy (đặt CL_CHAY_CODE=1)"
    with tempfile.TemporaryDirectory() as d:
        p = Path(d) / "bai.py"
        p.write_text(code + "\n\n" + test, encoding="utf-8")
        try:
            r = subprocess.run([sys.executable, "-I", str(p)], capture_output=True, text=True, timeout=30, cwd=d)
        except subprocess.TimeoutExpired:
            return False, "quá 30 giây"
        return r.returncode == 0, (r.stdout + r.stderr).strip()[-300:]


BAI_CODE = [
    ("code gộp khoảng", False,
     "Viết hàm Python `gop_khoang(ds)` nhận danh sách các khoảng [a, b] (số nguyên, a <= b, chưa sắp xếp) và trả danh sách "
     "các khoảng [a, b] sau khi gộp những khoảng chồng nhau hoặc chạm nhau, sắp tăng dần. Chỉ trả một khối code.",
     "f = lambda x: [list(k) for k in gop_khoang(x)]\n"
     "assert f([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]\nassert f([[1,4],[4,5]]) == [[1,5]]\n"
     "assert f([]) == []\nassert f([[5,7],[1,2],[2,3]]) == [[1,3],[5,7]]\nassert f([[1,10],[2,3]]) == [[1,10]]\nprint('OK')"),
    ("code LRU cache", False,
     "Viết lớp Python `LRUCache(capacity)` có `get(key)` (trả -1 nếu không có) và `put(key, value)`, cả hai O(1); khi đầy thì "
     "bỏ phần tử lâu chưa dùng nhất. Chỉ trả một khối code.",
     "c = LRUCache(2); c.put(1, 1); c.put(2, 2); assert c.get(1) == 1; c.put(3, 3); assert c.get(2) == -1\n"
     "c.put(4, 4); assert c.get(1) == -1; assert c.get(3) == 3; assert c.get(4) == 4\n"
     "d = LRUCache(2); d.put(1, 1); d.put(2, 2); d.put(1, 10); d.put(3, 3); assert d.get(1) == 10; assert d.get(2) == -1\nprint('OK')"),
    ("code tính biểu thức (có nghĩ)", True,
     "Viết hàm Python `tinh(bt: str) -> float` tính giá trị biểu thức số học gồm số thực, + - * /, ngoặc đơn, dấu cách và dấu "
     "trừ một ngôi, đúng thứ tự ưu tiên. Không dùng eval, exec, compile hay thư viện ngoài. Chỉ trả một khối code.",
     "for bt, kq in [('2+3*4', 14), ('(1+2)*-3', -9), ('10/4', 2.5), ('-(2+3)*2', -10), (' 7 - 2 - 1 ', 4),\n"
     "               ('2*(3+(4-1))/3', 4), ('1.5*2', 3), ('--3', 3), ('8/2/2', 2)]:\n"
     "    assert abs(tinh(bt) - kq) < 1e-9, (bt, tinh(bt), kq)\nprint('OK')"),
]
BAI_TOAN = [
    ("toán xếp hàng (có nghĩ)", "Có bao nhiêu cách xếp 5 người A, B, C, D, E thành một hàng sao cho A và B không đứng cạnh nhau? "
     "Dòng cuối chỉ ghi đáp số.", "72"),
    ("toán vòi nước (có nghĩ)", "Một bể có hai vòi chảy vào: vòi 1 chảy riêng đầy bể sau 6 giờ, vòi 2 sau 4 giờ; vòi xả chảy riêng "
     "cạn bể đầy sau 12 giờ. Bể đang cạn, mở cả ba vòi cùng lúc thì sau bao nhiêu giờ đầy bể? Dòng cuối chỉ ghi đáp số.", "3"),
]
CONG_CU = [
    {"type": "function", "function": {"name": "get_weather", "description": "Lấy nhiệt độ hiện tại (độ C) của một thành phố",
                                      "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "doi_tien", "description": "Đổi một số tiền từ loại tiền này sang loại khác theo tỷ giá hôm nay",
                                      "parameters": {"type": "object", "properties": {"so_tien": {"type": "number"}, "tu": {"type": "string"},
                                                                                      "sang": {"type": "string"}}, "required": ["so_tien", "tu", "sang"]}}}]

# OCR phụ đề: khung hình vẽ tại chỗ (nền rối nhòe, chữ đậm viền đen như phụ đề cứng, lưu JPEG như khung video), đáp án là chữ
# đã vẽ. Font Noto Sans SC (giấy phép OFL) ghim theo commit của google/fonts, kiểm sha256; không tải được thì bỏ phần OCR.
FONT_TRUNG = ("https://raw.githubusercontent.com/google/fonts/2894aab31764f10f29c421bdfd2340d3b382d384/ofl/notosanssc/"
              "NotoSansSC%5Bwght%5D.ttf", "a3041811a78c361b1de50f953c805e0244951c21c5bd412f7232ef0d899af0da")
# (tên, cỡ khung, các dòng phụ đề, cỡ chữ, màu, chữ lạ ở trên cùng)
BAI_OCR = [
    ("ngang, chữ trắng", (1280, 720), ["妈，你别担心，我下个月就回家看你。"], 44, "white", None),
    ("cổ trang, chữ vàng", (1280, 720), ["王爷，属下办事不力，请您责罚！"], 40, (255, 222, 60), None),
    ("dọc, hai dòng, có tiêu đề trên", (720, 1280), ["这台扫地机器人把小哈", "吓得直往沙发底下钻"], 40, "white", "我的哈士奇变成美女了 第13集"),
    ("tên game, dấu lạ", (1280, 720), ["旅行者，你终于来了！", "派蒙等你好久了～"], 42, "white", None),
    ("chữ nhỏ, có số", (1280, 720), ["芯片短缺让全球汽车厂商损失惨重，预计2026年才会缓解"], 26, (255, 230, 80), None),
    ("phồn thể", (1280, 720), ["這個問題我們明天再討論吧"], 36, "white", None),
]
# Dịch từng câu: (ngữ cảnh, câu gốc, các nhóm phải có (regex, không phân biệt hoa thường), regex cấm). Máy chỉ kiểm thô;
# câu dịch in ra cuối để đọc chấm.
BAI_DICH = [
    ("phim gia đình, con trai nói với mẹ qua điện thoại", "妈，你别担心，我下个月就回家看你。",
     [r"\bmẹ\b", r"\bcon\b"], r"\b(tôi|bạn)\b"),
    ("phim cổ trang, thị vệ quỳ trước vương gia", "王爷，属下办事不力，请您责罚。",
     [r"vương gia", r"thuộc hạ"], r"\b(tôi|bạn)\b"),
    ("phim tổng tài, thư ký báo cáo với sếp", "总裁，这份合同对方已经签字了。",
     [r"tổng tài|tổng giám đốc|giám đốc|sếp", r"hợp đồng", r"\bký\b|\bkí\b"], None),
    ("video khoa học phổ thông, câu mở đầu", "一直往地底下挖，会发现什么？", [r"đào", r"đất"], None),
    ("video AI: chú husky tên Tiểu Ha biến thành cô gái", "小哈最怕扫地机器人了！",
     [r"tiểu ha", r"robot|rô-bốt|máy hút bụi|hút bụi", r"sợ"], None),
    ("game Genshin Impact", "旅行者，你终于来了，派蒙等你好久了！", [r"lữ hành|lữ khách", r"paimon"], None),
    ("phim, hai người đàn ông cãi nhau gay gắt", "你给我等着，这笔账我迟早跟你算！",
     [r"chờ|đợi", r"tính|thanh toán|trả"], r"\bbạn\b"),
    ("tin công nghệ", "芯片短缺让全球汽车厂商损失惨重。", [r"chip", r"ô tô|xe hơi", r"thiếu"], None),
    ("reels hài, cô gái nói với bạn trai", "I promise I'm always this attentive.", [r"hứa"], None),
    ("reels cosplay Wuthering Waves, bị chê giữa công viên", "I'm getting flamed in public now, all I wanted was a peaceful "
     "afternoon at the park.", [r"công viên", r"chiều"], None),
]
BAI_DICH_LO = ["儿子，你怎么又这么晚才回来？", "妈，公司临时加班，我也没办法。", "你看看你，瘦成什么样了。",
               "我没事，您别操心了。", "饭在锅里热着呢，快去吃吧。", "谢谢妈，还是家里的饭最香。"]
# Việc làm nội dung: (tên, đề, hàm kiểm thô trên câu trả lời -> (đạt, ghi chú))
_dong = lambda t: [d.strip(" -*•\t") for d in t.splitlines() if d.strip(" -*•\t")]
BAI_NOI_DUNG = [
    ("tiêu đề YouTube", "Viết 8 tiêu đề YouTube tiếng Việt cho video giải thích: pin natri có thay được pin lithium trên ô tô điện "
     "không. Mỗi tiêu đề dưới 60 ký tự, gây tò mò nhưng không nói sai sự thật. Mỗi dòng một tiêu đề, không đánh số, không giải thích.",
     lambda t: (len(_dong(t)) >= 8 and all(len(d) <= 75 for d in _dong(t)[:8]), f"{len(_dong(t))} dòng, dài nhất "
                f"{max((len(d) for d in _dong(t)), default=0)} ký tự")),
    ("lời mở đầu 30 giây", "Viết lời mở đầu 30 giây (70–100 từ) cho video 'Sẽ ra sao nếu Trái Đất ngừng quay trong 1 giây'. "
     "Câu đầu phải gây sốc bằng một cảnh cụ thể. Chỉ trả lời bằng lời đọc.",
     lambda t: (50 <= len(t.split()) <= 140, f"{len(t.split())} từ")),
    ("lời bình shorts game", "Viết 3 câu bình luận hài, mỗi câu dưới 15 từ, để chèn vào shorts game Genshin Impact: Furina và "
     "Nahida nhìn nhau rồi cùng quay mặt đi. Mỗi dòng một câu, không đánh số.",
     lambda t: (len(_dong(t)) == 3 and all(len(d.split()) <= 18 for d in _dong(t)), f"{len(_dong(t))} dòng")),
    ("mô tả + thẻ", "Viết mô tả YouTube khoảng 200 ký tự, có từ khóa, cho video giải thích pin natri. Dòng cuối là đúng 5 thẻ, "
     "cách nhau bằng dấu phẩy. Không dùng hashtag.",
     lambda t: (bool(_dong(t)) and len(_dong(t)[-1].split(",")) == 5 and "#" not in t and len(t) <= 600,
                f"{len(t)} ký tự, dòng cuối {len(_dong(t)[-1].split(',')) if _dong(t) else 0} thẻ")),
    ("biết nói không biết", "Doanh thu của cửa hàng Bách Hóa Xanh ở phường Bến Nghé, TP.HCM ngày 3/9/2026 là bao nhiêu?",
     lambda t: (bool(re.search(r"không (có|biết|thể|nắm|truy cập)|chưa có|không rõ", t.lower())), "phải nói không biết, không bịa số")),
]


def font_trung():
    p = Path(tempfile.gettempdir()) / "NotoSansSC-wght.ttf"
    if p.exists() and hashlib.sha256(p.read_bytes()).hexdigest() == FONT_TRUNG[1]:
        return p
    r = requests.get(FONT_TRUNG[0], timeout=180)
    r.raise_for_status()
    if hashlib.sha256(r.content).hexdigest() != FONT_TRUNG[1]:
        raise RuntimeError("mã băm font không khớp")
    p.write_bytes(r.content)
    return p


def khung_phu_de(font, co_khung, dong, co, mau, tren, seed):
    """Một khung hình giả: nền chuyển màu + vòng tròn màu nhòe, phụ đề đậm viền đen ở dưới, trả data URL JPEG."""
    from PIL import Image, ImageDraw, ImageFilter, ImageFont
    rnd = random.Random(seed)
    w, h = co_khung
    img = Image.new("RGB", (w, h))
    dr = ImageDraw.Draw(img)
    c1, c2 = [tuple(rnd.randint(30, 220) for _ in range(3)) for _ in range(2)]
    for y in range(h):
        dr.line([(0, y), (w, y)], fill=tuple(int(a + (b - a) * y / h) for a, b in zip(c1, c2)))
    for _ in range(30):
        x, y, r = rnd.randint(0, w), rnd.randint(0, h), rnd.randint(20, 220)
        dr.ellipse((x - r, y - r, x + r, y + r), fill=tuple(rnd.randint(0, 255) for _ in range(3)))
    img = img.filter(ImageFilter.GaussianBlur(5))
    dr = ImageDraw.Draw(img)
    f = ImageFont.truetype(str(font), co)
    try:
        f.set_variation_by_name("Bold")
    except Exception:
        pass
    y = h - (h // 7 if h > w else 40) - (co + 12) * len(dong)
    for d in dong:
        b = dr.textbbox((0, 0), d, font=f, stroke_width=3)
        dr.text(((w - (b[2] - b[0])) // 2, y), d, font=f, fill=mau, stroke_width=3, stroke_fill="black")
        y += co + 12
    if tren:
        f2 = ImageFont.truetype(str(font), 34)
        b = dr.textbbox((0, 0), tren, font=f2)
        dr.text(((w - (b[2] - b[0])) // 2, 60), tren, font=f2, fill="white", stroke_width=2, stroke_fill="black")
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=75)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def chuan_chu(s):
    """Bỏ dấu câu, khoảng trắng; giữ chữ Hán và chữ số, chữ cái Latin (đổi chữ toàn khổ về thường)."""
    s = unicodedata.normalize("NFKC", s)
    return "".join(ch for ch in s if CJK.match(ch) or (ch.isascii() and ch.isalnum()))


def ti_le_sai(doan, dung):
    """Tỉ lệ sai chữ (CER): khoảng cách sửa chữ / số chữ đáp án."""
    a, b = chuan_chu(doan), chuan_chu(dung)
    truoc = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        hien = [i]
        for j, cb in enumerate(b, 1):
            hien.append(min(truoc[j] + 1, hien[j - 1] + 1, truoc[j - 1] + (ca != cb)))
        truoc = hien
    return truoc[-1] / max(len(b), 1)


def kiem_chat_luong():
    KQ, CHU = [], []

    def ghi(muc, dat, chi_tiet, r=None):
        toc = f" | {r['n']} token, {r['tok_s']:.1f} token/giây, {r['giay']:.1f} giây" if r and r.get("n") else ""
        KQ.append((muc, dat))
        print(f"  {muc:<32} {'ĐẠT' if dat else ('—' if dat is None else 'SAI')}  {chi_tiet}{toc}", flush=True)

    for _ten in ("ENGINE_PROC", "gw_proc"):
        if _g.get(_ten) is not None and _g[_ten].poll() is not None:
            print(f"{_ten} đã tắt: chạy lại từ ô 6 rồi mới kiểm chất lượng.")
            return
    print(f"Kiểm chất lượng {CL_MODEL} qua {CL_URL} (đọc ảnh: {'có' if CL_ANH else 'không'}, chạy code: {'có' if CL_CHAY_CODE else 'không'})",
          flush=True)
    for muc, nghi, de, test in BAI_CODE:
        lan, loi, r = [], "", None
        for _ in range(CL_LAP):
            try:
                r = goi([{"role": "user", "content": de}], nghi=nghi, muc="medium", max_tokens=12000 if nghi else 3000)
                code = lay_code(r["noi"])
                if re.search(r"\b(eval|exec|compile)\s*\(", code) and "tinh" in de:
                    ok, out = False, "dùng eval/exec dù đề cấm"
                else:
                    ok, out = chay_code(code, test)
            except Exception as e:
                ok, out = False, f"LỖI {type(e).__name__}: {str(e)[:150]}"
            lan.append(ok)
            loi = loi or ("" if ok else out)
        so = sum(1 for x in lan if x)
        ghi(muc, None if all(x is None for x in lan) else so * 2 > len(lan),
            f"đúng {so}/{len(lan)} lần" + (f"; lần hỏng đầu: {loi[-140:]!r}" if loi else ""), r)
    for muc, de, dap in BAI_TOAN:
        try:
            r = goi([{"role": "user", "content": de}], nghi=True, muc="low", max_tokens=12000)
            dong = r["noi"].splitlines()[-1] if r["noi"] else ""
            so = re.findall(r"-?\d+(?:[.,]\d+)?", dong)
            ghi(muc, bool(so) and so[-1].replace(",", ".") in (dap, dap + ".0"),
                f"đáp {dong[:60]!r} (đúng {dap}), nghĩ {len(r['nghi'])} ký tự" if dong else "không trả lời", r)
        except Exception as e:
            ghi(muc, False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    # gọi tool hai lượt: model gọi tool, nhận kết quả, rồi trả lời bằng số liệu đó
    try:
        msgs = [{"role": "user", "content": "Hà Nội đang bao nhiêu độ, và 100 USD đổi được bao nhiêu VND? Dùng công cụ để tra."}]
        goi_ten, r = [], None
        for _ in range(4):
            r = goi(msgs, max_tokens=1500, tools=CONG_CU)
            if not r["tool_calls"]:
                break
            msgs.append({"role": "assistant", "content": r["noi"] or None, "tool_calls": [
                {"id": tc["id"] or f"call_{i}", "type": "function", "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for i, tc in enumerate(r["tool_calls"])]})
            for i, tc in enumerate(r["tool_calls"]):
                goi_ten.append(tc["name"])
                kq = {"get_weather": {"city": "Hà Nội", "nhiet_do_c": 31},
                      "doi_tien": {"ket_qua": 2617500, "loai": "VND"}}.get(tc["name"], {"loi": "không có công cụ này"})
                msgs.append({"role": "tool", "tool_call_id": tc["id"] or f"call_{i}", "content": json.dumps(kq, ensure_ascii=False)})
        cuoi = (r or {}).get("noi", "")
        dat = {"get_weather", "doi_tien"} <= set(goi_ten) and "31" in cuoi and "2617500" in re.sub(r"[.,\s]", "", cuoi)
        ghi("gọi tool hai lượt", dat, f"đã gọi {goi_ten}; trả: {cuoi[:90]!r}", r)
    except Exception as e:
        ghi("gọi tool hai lượt", False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    # đọc ảnh: hình và chữ
    if CL_ANH:
        try:
            from PIL import Image, ImageDraw, ImageFont
            img = Image.new("RGB", (800, 400), "white")
            dr = ImageDraw.Draw(img)
            dr.ellipse((60, 110, 240, 290), fill=(220, 30, 30))
            dr.rectangle((560, 110, 740, 290), fill=(30, 60, 220))
            try:
                font = ImageFont.load_default(size=56)
            except TypeError:
                font = ImageFont.load_default()
            dr.text((300, 170), "APPLE 42", fill="black", font=font)
            buf = io.BytesIO()
            img.save(buf, "PNG")
            url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
            r = goi([{"role": "user", "content": [{"type": "image_url", "image_url": {"url": url}}, {"type": "text", "text":
                     "Mô tả ngắn ảnh này: có những hình gì, màu gì, nằm bên trái hay bên phải, và chữ trong ảnh là gì?"}]}], max_tokens=600)
            t = r["noi"].lower()
            dat = ("đỏ" in t or "red" in t) and ("xanh" in t or "blue" in t) and "42" in t and "apple" in t
            ghi("đọc ảnh (hình, màu, chữ)", dat, f"{r['noi'][:140]!r}", r)
        except Exception as e:
            ghi("đọc ảnh (hình, màu, chữ)", False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    # ngữ cảnh dài: tìm một dòng giữa khoảng CL_DAI token
    try:
        dong = [f"Dòng {i}: kho số {i % 50} nhận {i * 7 % 1000} thùng hàng vào ngày {i % 28 + 1}." for i in range(CL_DAI // 22)]
        dong.insert(int(len(dong) * 0.6), "Ghi chú quan trọng: mật mã mở cửa kho số 77 là 8137.")
        r = goi([{"role": "user", "content": "\n".join(dong) + "\n\nMật mã mở cửa kho số 77 là gì? Chỉ trả lời con số."}], max_tokens=64)
        ghi(f"ngữ cảnh dài (~{CL_DAI // 1000}K token)", "8137" in r["noi"],
            f"trả {r['noi'][:40]!r}, {r['tok_de']} token đề, chờ chữ đầu {r['cho_dau'] or 0:.1f} giây", r)
    except Exception as e:
        ghi(f"ngữ cảnh dài (~{CL_DAI // 1000}K token)", False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    # văn tiếng Việt: máy chỉ kiểm độ dài, dấu, chữ Hán lẫn vào; hay dở do người đọc
    try:
        r = goi([{"role": "user", "content": "Viết một đoạn văn khoảng 150 từ, giọng gần gũi, giải thích cho bố mẹ lớn tuổi vì sao nên "
                  "bật xác thực hai lớp cho tài khoản Facebook và cách bật."}], max_tokens=1200)
        so_tu = len(r["noi"].split())
        dau = len(re.findall(r"[ạảấầẩẫậắằẳẵặẹẻẽếềểễệịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹđ]", r["noi"].lower()))
        dat = 90 <= so_tu <= 260 and not CJK.search(r["noi"]) and dau > 20
        ghi("văn tiếng Việt", dat, f"{so_tu} từ, {'có' if CJK.search(r['noi']) else 'không'} lẫn chữ Hán", r)
        CHU.append(("Đoạn văn tiếng Việt (tự đọc chấm)", r["noi"]))
    except Exception as e:
        ghi("văn tiếng Việt", False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    # ---- Phần làm nội dung ----
    print("\nLàm nội dung (OCR phụ đề, dịch, viết):", flush=True)
    if CL_ANH:
        try:
            font = font_trung()
        except Exception as e:
            font = None
            ghi("OCR phụ đề", None, f"bỏ qua: không tải được font chữ Trung ({type(e).__name__}: {str(e)[:100]})")
        for i, (ten, co_khung, dong, co, mau, tren) in enumerate(BAI_OCR if font else []):
            muc = f"OCR: {ten}"
            try:
                url = khung_phu_de(font, co_khung, dong, co, mau, tren, seed=i + 1)
                r = goi([{"role": "user", "content": [{"type": "image_url", "image_url": {"url": url}}, {"type": "text", "text":
                         "Ảnh là một khung hình video. Chép nguyên văn phụ đề tiếng Trung ở phần dưới khung hình (bỏ qua chữ ở chỗ "
                         "khác). Giữ đúng chữ giản thể hay phồn thể như trong ảnh, không dịch, không giải thích, chỉ trả về chữ đó."}]}],
                        max_tokens=200)
                doc = "".join(l for l in r["noi"].splitlines() if CJK.search(l))
                sai = ti_le_sai(doc, "".join(dong))
                ghi(muc, sai <= 0.05, f"sai {sai:.0%} chữ; đọc {doc[:40]!r}", r)
            except Exception as e:
                ghi(muc, False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    dich = []
    for ngu_canh, cau, can, cam in BAI_DICH:
        muc = f"dịch: {ngu_canh.split(',')[0][:22]}"
        try:
            r = goi([{"role": "user", "content": f"Dịch câu phụ đề sau sang tiếng Việt tự nhiên, đúng xưng hô theo ngữ cảnh.\n"
                      f"Ngữ cảnh: {ngu_canh}.\nCâu: {cau}\nChỉ trả về câu dịch."}], max_tokens=300)
            t = r["noi"].strip()
            thieu = [c for c in can if not re.search(c, t, re.I)]
            pham = bool(cam and re.search(cam, t, re.I))
            ghi(muc, not thieu and not pham and not CJK.search(t),
                ("thiếu " + ", ".join(thieu) + "; " if thieu else "") + ("sai xưng hô; " if pham else "")
                + ("lẫn chữ Hán; " if CJK.search(t) else "") + f"{t[:70]!r}", r)
            dich.append(f"{cau}\n  → {t}")
        except Exception as e:
            ghi(muc, False, f"LỖI {type(e).__name__}: {str(e)[:150]}")
    try:
        de = "\n".join(f"{i}. {c}" for i, c in enumerate(BAI_DICH_LO, 1))
        r = goi([{"role": "user", "content": "Dịch lô phụ đề sau sang tiếng Việt (phim gia đình: mẹ và con trai đã đi làm). Giữ đúng "
                  f"{len(BAI_DICH_LO)} dòng, mỗi dòng bắt đầu bằng số thứ tự như bản gốc, đúng xưng hô mẹ – con, không thêm gì khác.\n\n{de}"}],
                max_tokens=800)
        so_dong = re.findall(r"(?m)^\s*(\d+)[.)]\s*\S", r["noi"])
        dat = so_dong == [str(i) for i in range(1, len(BAI_DICH_LO) + 1)] and not CJK.search(r["noi"]) \
            and not re.search(r"\b(tôi|bạn)\b", r["noi"], re.I)
        ghi("dịch lô 6 dòng đánh số", dat, f"ra {len(so_dong)} dòng đánh số", r)
        dich.append(f"[lô 6 dòng]\n{r['noi']}")
    except Exception as e:
        ghi("dịch lô 6 dòng đánh số", False, f"LỖI {type(e).__name__}: {str(e)[:150]}")
    if dich:
        CHU.append(("Câu dịch (tự đọc chấm)", "\n".join(dich)))

    for ten, de, kiem in BAI_NOI_DUNG:
        try:
            r = goi([{"role": "user", "content": de}], max_tokens=900)
            dat, ghi_chu = kiem(r["noi"])
            ghi(ten, dat and not CJK.search(r["noi"]), ghi_chu + ("; lẫn chữ Hán" if CJK.search(r["noi"]) else ""), r)
            CHU.append((ten.capitalize() + " (tự đọc chấm)", r["noi"]))
        except Exception as e:
            ghi(ten, False, f"LỖI {type(e).__name__}: {str(e)[:150]}")

    cham = [d for _, d in KQ if d is not None]
    print(f"\nMáy chấm: đạt {sum(cham)}/{len(cham)} mục.")
    for tieu_de, chu in CHU:
        print(f"\n--- {tieu_de} ---\n{chu}")


if _g.get("KIEM_CHAT_LUONG", True):
    try:
        kiem_chat_luong()
    except Exception as e:      # ô này hỏng không được làm dừng ô 10 (giữ phiên) khi bấm Run all
        print(f"Kiểm chất lượng dừng giữa chừng: {type(e).__name__}: {str(e)[:200]}")
else:
    print("Bỏ qua kiểm chất lượng (KIEM_CHAT_LUONG = False ở ô 2).")

In [ ]:
# 10) Giữ phiên sống và báo nhịp. Bấm Stop ở ô này KHÔNG tắt endpoint; muốn tắt hẳn thì chạy tat_het().
def tat_het():
    for ten, p in (("tunnel", cf_proc), ("gateway", gw_proc), (ENGINE_TEN, ENGINE_PROC)):
        if p is not None and p.poll() is None:
            p.terminate()
            print("đã tắt", ten)


def gio_vn():
    return time.strftime("%H:%M", time.gmtime(time.time() + 7 * 3600))


_het = time.time() + KEEP_ALIVE_HOURS * 3600
print(f"{gio_vn()} (giờ VN): giữ endpoint {KEEP_ALIVE_HOURS} giờ rồi tự tắt. Sửa KEEP_ALIVE_HOURS ở ô 2 nếu cần.")
try:
    while time.time() < _het:
        _chet = [t for t, p in ((ENGINE_TEN, ENGINE_PROC), ("gateway", gw_proc), ("tunnel", cf_proc))
                 if p is not None and p.poll() is not None]
        if _chet:
            print(f"{gio_vn()} ĐÃ CHẾT: {_chet}. Xem log trong {LOG_DIR}, rồi chạy lại từ ô 6.")
            break
        _luot = gw_log.read_text(errors="ignore").count("POST /v1/chat/completions")
        _gpu = " ".join(f"GPU{i}:{u}MiB/{p_}%" for i, u, p_ in gpu_csv("index,memory.used,utilization.gpu"))
        print(f"{gio_vn()} | còn {(_het - time.time()) / 3600:.1f} giờ | {_luot} lượt chat | {_gpu}", flush=True)
        time.sleep(min(300, max(1, _het - time.time())))
    else:
        print(f"{gio_vn()} hết {KEEP_ALIVE_HOURS} giờ, tắt endpoint.")
        tat_het()
except KeyboardInterrupt:
    print("Đã dừng ô báo nhịp. Endpoint VẪN chạy. Tắt hẳn: chạy tat_het() trong một ô mới.")